In [1]:
!git clone https://github.com/Vinuitik/TypiClustImplement.git
%cd TypiClustImplement

Cloning into 'TypiClustImplement'...
remote: Enumerating objects: 486, done.
remote: Total 486 (delta 0), reused 0 (delta 0), pack-reused 486 (from 1)
Receiving objects: 100% (486/486), 109.79 MiB | 37.45 MiB/s, done.
Resolving deltas: 100% (348/348), done.
/kaggle/working/TypiClustImplement


In [ ]:
# faiss-cpu for KMeans; KNN uses PyTorch (GPU via torch.cuda)
!pip install -q faiss-cpu

import faiss
import torch
print(f"FAISS version : {faiss.__version__}")
print(f"PyTorch CUDA  : {torch.cuda.is_available()}  |  device count: {torch.cuda.device_count()}")

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import torch

_ROOT = Path.cwd()
_DEEPAL_DIR = _ROOT / 'deep-active-learning'
if str(_DEEPAL_DIR) not in sys.path:
    sys.path.insert(0, str(_DEEPAL_DIR))

from data import get_CIFAR10
from handlers import CIFAR10_Handler
from utils import get_net

from al_methods import STRATEGY_REGISTRY, select_indices, typiclust, typiclust_improv

BUDGETS = [10, 20, 30, 40, 50, 60, 100, 150, 200, 250, 300]
METHODS = sorted(STRATEGY_REGISTRY.keys()) + ['typiclust', 'typiclust_improv']
EMBEDDINGS_PATH = str(_ROOT / 'datasets' / 'cifar10_train_embeddings.npz')
N_CLASSES = 10  # CIFAR-10
OUTPUT_DIR = _ROOT / 'datasets' / 'al_splits'
SEED = 42

print('CUDA available:', torch.cuda.is_available())
print('Methods:', METHODS)

CUDA available: False
Methods: ['bald_dropout', 'entropy', 'entropy_dropout', 'kcenter', 'kmeans', 'least_confidence', 'least_confidence_dropout', 'margin', 'margin_dropout', 'random', 'typiclust', 'typiclust_improv']


In [3]:
import time
import logging

logging.basicConfig(
    level=logging.INFO,
    format='[%(asctime)s] %(levelname)s - %(message)s',
    datefmt='%H:%M:%S'
)

def _sorted_ints(values):
    return sorted(int(v) for v in values)


def _run_method(method: str) -> None:
    budgets = sorted(BUDGETS)
    total_budgets = len(budgets)

    np.random.seed(SEED)
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

    logging.info(f'Start method={method} | budgets={budgets}')

    dataset = get_CIFAR10(CIFAR10_Handler)

    if method in ('typiclust', 'typiclust_improv'):
        logging.info('Embedding-based method: no initial labeled set required')
    else:
        logging.info('Initializing with 1 sample per class')
        y = dataset.Y_train.numpy()
        for cls in range(N_CLASSES):
            cls_idxs = np.where(y == cls)[0]
            chosen = np.random.choice(cls_idxs, 1, replace=False)
            dataset.labeled_idxs[chosen] = True

    labeled = np.where(dataset.labeled_idxs)[0].tolist()
    unlabeled = np.where(~dataset.labeled_idxs)[0].tolist()

    logging.info(f'Initial labeled={len(labeled)} | unlabeled={len(unlabeled)}')

    net = None
    if method in STRATEGY_REGISTRY and method != 'random':
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        net = get_net('CIFAR10', device)
        logging.info(f'Using strategy model on device={device}')

    current_labeled_count = len(labeled)
    method_start = time.time()

    for i, budget in enumerate(budgets, 1):
        iter_start = time.time()

        increment = budget - current_labeled_count
        logging.info(
            f'[{method}] Budget {i}/{total_budgets} | target={budget} | '
            f'current={current_labeled_count} | +{increment}'
        )

        selected = []

        if increment > 0:
            if method in STRATEGY_REGISTRY:
                selected, unlabeled = select_indices(
                    dataset=dataset,
                    budget=increment,
                    strategy=method,
                    net=net,
                    train_before_query=(net is not None),
                    update_dataset_labels=True,
                )
                labeled = np.where(dataset.labeled_idxs)[0].tolist()

            elif method == 'typiclust':
                selected, unlabeled = typiclust(
                    dataset=dataset,
                    budget=budget,
                    embeddings_npz_path=EMBEDDINGS_PATH,
                )
                for idx in selected:
                    dataset.labeled_idxs[idx] = True
                labeled = np.where(dataset.labeled_idxs)[0].tolist()

            elif method == 'typiclust_improv':
                selected, unlabeled = typiclust_improv(
                    dataset=dataset,
                    budget=budget,
                    embeddings_npz_path=EMBEDDINGS_PATH,
                )
                for idx in selected:
                    dataset.labeled_idxs[idx] = True
                labeled = np.where(dataset.labeled_idxs)[0].tolist()

            current_labeled_count = len(labeled)

        iter_time = time.time() - iter_start
        elapsed = time.time() - method_start
        avg_time = elapsed / i
        eta = avg_time * (total_budgets - i)

        logging.info(
            f'[{method}] Done budget={budget} | selected={len(selected)} | '
            f'labeled={len(labeled)} | iter_time={iter_time:.1f}s | '
            f'ETA≈{eta/60:.1f} min'
        )

        out_path = OUTPUT_DIR / f'{method}_{budget}.json'
        out_path.write_text(
            json.dumps({
                'labeled_indices': _sorted_ints(labeled),
                'unlabeled_indices': _sorted_ints(unlabeled),
            }),
            encoding='utf-8',
        )

        logging.info(f'Saved {out_path.name}')

    total_time = time.time() - method_start
    logging.info(f'Finished method={method} in {total_time/60:.1f} min')

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for method in METHODS:
    if method != 'typiclust' and method != 'typiclust_improv':
        continue
    print(f'[{method}]')
    _run_method(method)
print('Done.')

[10:07:47] INFO - Start method=typiclust | budgets=[10, 20, 30, 40, 50, 60, 100, 150, 200, 250, 300]


[typiclust]


100%|██████████| 170M/170M [00:06<00:00, 25.0MB/s] 
[10:07:59] INFO - Embedding-based method: no initial labeled set required
[10:07:59] INFO - Initial labeled=0 | unlabeled=40000
[10:07:59] INFO - [typiclust] Budget 1/11 | target=10 | current=0 | +10


In [ ]:
# Zip all generated splits for easy download
import shutil
shutil.make_archive('al_splits', 'zip', OUTPUT_DIR)
print('al_splits.zip ready for download')